<a href="https://colab.research.google.com/github/soham-never-codes/Festiva-AI-Event-Planner/blob/main/notebooks/Festiva_Week4_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Mount + install ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os

PROJECT_DIR = '/content/drive/MyDrive/festiva'

# The essential safety net
os.makedirs(f'{PROJECT_DIR}/data/raw', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data/processed', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/rag', exist_ok=True)

print("⬇️ Installing LangChain & FAISS...")
!pip install -q langchain langchain-community sentence-transformers faiss-cpu
print("✅ Environment ready!")

In [ ]:
# Define knowledge documents ──────────────────────────────────
# These are the synthetic event planning guides
DOCUMENTS = [
    {
        "title": "Complete Wedding Planning Guide",
        "content": """A wedding in India typically requires 6-12 months of planning for large events.
Timeline for a 200-guest wedding:
- 6 months before: Book venue, shortlist caterer, set overall budget
- 4 months before: Finalise caterer, book photographer and videographer
- 3 months before: Book decorator, confirm entertainment (DJ/band)
- 2 months before: Send invitations, finalise menu, book hotel blocks for guests
- 1 month before: Confirm all vendors, final headcount, schedule rehearsal
- 1 week before: Final payments, coordinate logistics, confirm timelines
Budget distribution for Indian weddings:
Venue: 25-35%, Catering: 20-30%, Decoration: 10-18%
Photography: 8-12%, Entertainment: 5-10%, Miscellaneous: 5-10%
Bangalore-specific: Peak season November-February. Book venues 6+ months ahead.
Average catering: ₹1200-₹2000 per plate. Popular areas: Kanakapura Road farmhouses."""
    },
    {
        "title": "Corporate Event Planning",
        "content": """Corporate events range from offsites (20 ppl) to large conferences (500+ ppl).
Budget distribution: Venue+AV 40-50%, Catering 20-28%, Speakers 5-15%, Marketing 5-10%.
Planning timeline for 100-person conference:
8 weeks: venue, 6 weeks: speakers+AV, 4 weeks: invitations, 2 weeks: rehearsal."""
    },
    {
        "title": "Birthday Party Planning",
        "content": """Birthday parties in Bangalore — budget ₹50,000 to ₹3,00,000 for 50 guests.
Breakdown: Venue 20-30%, Catering 25-35%, Cake 3-8%, Decoration 10-18%, Entertainment 8-15%.
4-week plan: Week1=venue+theme, Week2=caterer+cake+DJ, Week3=invites+decor, Week4=confirm."""
    },
    {
        "title": "Budget Optimization Strategies",
        "content": """Save 15-25% with smart planning:
1. Off-peak timing: Avoid Nov-Feb for weddings (30% premium). Weekdays 20-30% cheaper.
2. Venue negotiation: Get package deals. Compare 3-5 venues.
3. Catering: Buffet 20% cheaper than plated. Limit alcohol (adds 30-50%).
4. Decoration: Invest in entrance and stage (most photographed). Reuse across functions.
5. Bundle vendors: photographer+videographer same studio = 10-15% off.
6. Buffer: Keep 8-10% unallocated for last-minute expenses."""
    },
    {
        "title": "Vendor Selection Guide",
        "content": """Vendor evaluation: portfolio review, references (2-3 past clients), trial/tasting, written quotes x3, contract review.
Red flags: no written contract, 100% advance, vague packages, no backup plan.
Payment schedule: Venue 25/50/25%, Caterer 25/50/25%, Photographer 30/70%.
Coordinate: share venue layout, give all vendors each other's contacts, assign single point of contact per category."""
    }
]
print(f"✅ Loaded {len(DOCUMENTS)} knowledge documents")

In [ ]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

print("⚙️ Initializing text splitter and embedding model...")
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60)
embedder = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"}   # use GPU for fast embedding
)

chunks = []
for doc in DOCUMENTS:
    # [FIXED] Removed the '* 200'. LangChain automatically applies this metadata to all chunks.
    c = splitter.create_documents(
        [doc["content"]],
        metadatas=[{"title": doc["title"]}]
    )
    chunks.extend(c)

print(f"✂️ Sliced documents into {len(chunks)} chunks.")

print("🧠 Building FAISS vector index...")
vectorstore = FAISS.from_documents(chunks, embedder)

# Save the Vector Database to your Google Drive
faiss_path = f'{PROJECT_DIR}/rag/faiss_index'
vectorstore.save_local(faiss_path)
print(f"✅ FAISS index successfully saved to {faiss_path}!")

In [ ]:
#  Test retrieval ───────────────────────────────────────────────
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

test_queries = [
    "wedding timeline 200 guests Bangalore",
    "how to save money on catering",
    "corporate conference budget breakdown"
]

print("🔍 Testing Vector Database Retrieval:")
print("-" * 60)
for q in test_queries:
    print(f"\n🗣️ Query: '{q}'")
    docs = retriever.invoke(q)
    for i, d in enumerate(docs, 1):
        print(f"  [{i}] Source: {d.metadata.get('title','?')} \n      Snippet: {d.page_content[:120]}...")